# Gradio demo: OCR (simple)

In [ ]:
import gradio as gr

# We do the work in here
def greet(name):
    value = f"Hello, {name}!"

    return value

with gr.Blocks() as demo:
    gr.Markdown("## A simple Gradio app")

    with gr.Row():
        # These gr. bits are all of our user interface
        with gr.Column():
            name = gr.Textbox(label="Name")
            btn = gr.Button("Submit", variant='primary')
        with gr.Column():
            output = gr.Textbox(label="Output")

    # When the button is clicked...
    # run greet...
    # with the value of name as input
    # and put the result in output.
    btn.click(greet, inputs=name, outputs=output)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Let's say we're talking about **OCR** and I've been bragging about how good RapidOCR is instead of Tesseract.

You have some awful PDFs, you want to try them out. Why should you need to email them to me?

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import fitz
import gradio as gr
from rapidocr import RapidOCR


DPI = 200
ocr_engine = RapidOCR()


def render_pdf_pages(pdf_path: str, image_dir: Path) -> list[Path]:
    image_paths = []
    zoom = DPI / 72
    matrix = fitz.Matrix(zoom, zoom)

    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            pixmap = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = image_dir / f"page-{page_number}.png"
            pixmap.save(image_path)
            image_paths.append(image_path)

    return image_paths


def ocr_pdf(pdf_file: str) -> str:
    with TemporaryDirectory() as image_dir:
        image_paths = render_pdf_pages(pdf_file, Path(image_dir))

        page_texts = []
        for image_path in image_paths:
            result = ocr_engine(str(image_path))
            text = "\n".join(result.txts or "").strip()
            page_texts.append(text)

    return "\n\n".join(page_texts)


with gr.Blocks(title="PDF OCR demo") as demo:
    gr.Markdown("# PDF OCR demo")

    pdf = gr.File(label="PDF", file_types=[".pdf"], type="filepath")
    button = gr.Button("OCR PDF", variant="primary")
    output = gr.Textbox(label="OCR text", lines=24)

    button.click(ocr_pdf, pdf, output)


if __name__ == "__main__":
    demo.launch()


[INFO] 2026-05-24 18:08:46,162 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-24 18:08:46,237 [RapidOCR] download_file.py:60: File exists and is valid: /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-24 18:08:46,237 [RapidOCR] main.py:57: Using /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-24 18:08:46,308 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-24 18:08:46,310 [RapidOCR] download_file.py:60: File exists and is valid: /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-24 18:08:46,311 [RapidOCR] main.py:57: Using /Users/

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
